# Section 3: ELT with WebHDFS InsecureClient
Run `pip install hdfs pandas` once before starting.

## 1. Connect to HDFS and load data

In [ ]:
from hdfs import InsecureClient
import pandas as pd

# Connect to Hadoop sandbox (WebHDFS). If localhost fails, use the sandbox IP.
client = InsecureClient('http://localhost:50070', user='maria_dev')

# Read CSV from HDFS into a DataFrame
with client.read('/user/maria_dev/datasets/penguins.csv', encoding='utf-8') as reader:
    df = pd.read_csv(reader)

df.head()

## 2. Analyze structure and missing values

In [ ]:
df.info()
print("Shape:", df.shape)
print(df.isnull().sum())

## 3. Clean and preprocess

In [ ]:
# Drop rows with missing numeric measurements
df = df.dropna(subset=['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g'])

# Fill missing sex with 'unknown'
df['sex'] = df['sex'].fillna('unknown')

# Convert body mass from grams to kilograms
df['body_mass_kg'] = df['body_mass_g'] / 1000
df = df.drop(columns=['body_mass_g'])

print(df.isnull().sum())
df.head()

## 4. Upload cleaned data to HDFS

In [ ]:
client.makedirs('/user/maria_dev/datasets/clean')

with client.write('/user/maria_dev/datasets/clean/penguins_clean.csv', encoding='utf-8', overwrite=True) as writer:
    df.to_csv(writer, index=False)

print(client.list('/user/maria_dev/datasets/clean'))